# ⚡ Tool-call accuracy gate: per-step tool-call accuracy via SageMaker Custom Scorer evaluation

The full evaluation (`3-evaluate-shopping-agent.ipynb`) is expensive: it stands up a Lucene search server over 2.7M products, deploys endpoints, and runs a 30-step agent loop per problem. This notebook is a **cheap managed pre-check** you run *first*, using SageMaker's **Custom Scorer** model-evaluation capability.

**What it measures — teacher-forced, per-step tool-call accuracy.** We turn each held-out validation example (`{prompt, completion}` from the training notebook) into an evaluation row `{query, reference_answer}`. The managed evaluation job generates the student's single next action for each `query`, then calls our **custom reward function** (a Lambda registered as a SageMaker Evaluator) to compare it against the teacher's ground-truth action.

**Why use the managed capability here.** Per-step tool-call scoring is single-turn `query → response` with ground truth — exactly what Custom Scorer handles. We get managed generation, scoring, MLflow lineage, and optional base-vs-student comparison without hand-rolling an inference loop.

**Why this is a valid gate (and its limits).**
- It answers: *did distillation teach the model to emit the right tool with the right parameters, given correct context?*
- It is **not** end-to-end task success (ASR): it uses teacher-forced context, so it can't see error compounding or recovery. A model can pass this and still underperform on ASR — but one that **fails** this will almost certainly fail ASR.

**The decision rule.** If the reward (tool-set accuracy) clears your threshold → run the full ASR eval. If not → fix training first.

> **Note on fidelity:** the managed job renders each `query` through the model's default chat template and generation settings, so this is a close managed *equivalent* of per-step accuracy, not a byte-identical reproduction of a hand-driven chat loop.

## 0. Setup

Install the SDK, load the student `MODEL_PKG_ARN` from the training notebook (stored there via `%store`), and resolve the AWS session/role. Everything in this notebook is managed — there are no local servers or endpoints to stand up — so the only dependency is `sagemaker` + `boto3`.

In [ ]:
%pip install -q "sagemaker==3.23.0" "boto3==1.43.108"

In [ ]:
# The student Model Package ARN produced by the training notebook (step 03 did `%store MODEL_PKG_ARN`).
%store -r MODEL_PKG_ARN

## 1. Configuration

The knobs for the litmus run:

- **`VAL_FILE`** — the held-out `{prompt, completion}` rows from the training notebook (the teacher's ground-truth next actions).
- **`N_EXAMPLES`** — cap the eval to a random subset (keeps the managed job cheap/fast); `None` uses all held-out rows.
- **`MAX_QUERY_TOKENS`** — drop prompts longer than this; over-long prompts exceed the serving context window and fail the managed job.
- **`EVALUATE_BASE`** — also evaluate the base model (resolved from the Model Package's lineage) so every result has a base-vs-student comparison.
- **`PASS_THRESHOLD`** — the gate: proceed to the full ASR eval only if the student's tool-selection reward clears it.

In [ ]:
import os, sys
from pathlib import Path
import boto3

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = REGION

# litmus_utils (and the reward-function helpers) live in scripts/, next to this notebook — put it on sys.path
# so `from litmus_utils import ...` resolves. Run this notebook from its own folder (distillation/).
SCRIPTS = Path("scripts").resolve()
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from sagemaker.core.helper.session_helper import Session, get_execution_role
sagemaker_session = Session(boto_session=boto3.Session(region_name=REGION))
BUCKET = sagemaker_session.default_bucket()
ROLE = get_execution_role()


# ---- Config ----
VAL_FILE = "data/seqkd_val.jsonl"   # held-out {prompt, completion} from training (written to data/ by notebook 1)
N_EXAMPLES = 600                 # None = ALL held-out rows; an int takes a random subset before filtering/splitting
MAX_QUERY_TOKENS = 12000         # drop rows whose prompt is >= this many tokens (avoids max-context eval failures)
HF_TOKENIZER = "Qwen/Qwen3.5-4B" # tokenizer used to measure prompt length
PREFIX = "seqkd-shopping/litmus"
S3_OUTPUT = f"s3://{BUCKET}/{PREFIX}/output/"
EVALUATE_BASE = True             # also evaluate the base model (lineage-resolved) for a comparison row
PASS_THRESHOLD = 0.80            # proceed to full ASR eval if mean reward (tool-set accuracy) >= this
MLFLOW_EXPERIMENT_NAME = "seqkd-shopping-litmus-eval"

print(f"Region: {REGION}\nRole:{ROLE}\nModel Package ARN: {MODEL_PKG_ARN}")

## 2. Build the eval datasets (one per task)

Turn the local held-out rows into the Custom Scorer schema — `{"query": <prompt>, "response": <ground-truth action>, "system": <optional>}` — and upload one dataset per task so results break down by intent.

Two things happen here:
- **Task labelling.** `data/seqkd_val.jsonl` carries no task label, so we recover each row's intent (product/shop/voucher) by matching its `<user>…</user>` query against ShoppingBench's per-task test files. (Requires the ShoppingBench checkout from the ASR notebook's `setup_shopbench.sh`.) Rows that don't match any task are dropped from the per-task evals.
- **Length filter.** Prompts at/above `MAX_QUERY_TOKENS` are dropped — they exceed the serving context window and caused the managed job to fail.

In [ ]:
from litmus_utils import build_eval_datasets

EVAL_TASKS = ["product", "shop", "voucher"]
SB_DATA = Path("shopbench-eval/ShoppingBench/data")   # from the ASR notebook's setup_shopbench.sh checkout

# DATASETS: {task -> s3 uri}, COUNTS: {task -> row count}. See the section above for what this does.
DATASETS, COUNTS = build_eval_datasets(
    val_file=VAL_FILE, eval_tasks=EVAL_TASKS, sb_data=SB_DATA, tokenizer_id=HF_TOKENIZER,
    max_query_tokens=MAX_QUERY_TOKENS, n_examples=N_EXAMPLES,
    s3_prefix=PREFIX, bucket=BUCKET, region=REGION)

## 3. The custom reward function (scorer)

A **Custom Scorer** lets you grade a model with *your own* code instead of a built-in metric. You write a Python reward function, register it as an **Evaluator**, and the managed eval job runs it once per sample. Reach for one whenever "correct" is domain-specific — here, "did the model call the right tool?" — rather than a generic text-similarity score. (The sibling type, `RewardPrompt`, grades with an LLM judge instead of code; use `RewardFunction` when the check is deterministic, like ours.)

### Anatomy — the contract every reward function must satisfy

The registered file must expose two functions with these exact names:

- **`reward_function(sample, index) -> dict`** — the scorer. `sample` carries the model's output (`model_response`), the input (`query`), and the ground truth (`reference_answer`/`response`). It must return a dict with:
  - `aggregate_reward_score` (float) — the single number the job averages and the gate compares;
  - `metrics_list` — any number of extra named `{"name", "value", "type"}` metrics (`type` = `"Reward"` or `"Metric"`) for diagnostics that don't drive the gate;
  - `id` — a per-sample identifier.
- **`lambda_handler(event, context)`** — the AWS Lambda entry point. It unwraps the event, calls `reward_function`, and returns a JSON-wrapped result. This is boilerplate; keep it as-is.

Everything else in the file is yours.

### Anatomy of *this* scorer (`reward_functions/litmus_reward_function.py`)

Built bottom-up in four sections so each piece is testable:

1. **`parse_turn` / `_tool_names_lenient`** — normalize raw text into `(kind, [tool_names])`. A `strict` switch controls whether only the exact harness `<tool_call>[{...}]</tool_call>` array form counts, or looser forms (bare `{...}`, regex-extracted names) are accepted too.
2. **`_toolset`** — compare predicted tool names against the reference's → `1.0`/`0.0`.
3. **`reward_function`** — parse the reference (always strict), then score the prediction two ways and emit three graded metrics (below).
4. **`lambda_handler`** — the Lambda envelope.

### Design considerations (why it grades the way it does)

- **Grade in layers, gate on the one that matters.** Three levels, lenient → strict:
  | metric | measures | role |
  |---|---|---|
  | `kind_match` | right *kind* of turn (tool call vs. response)? | diagnostic |
  | `toolset_match` | right *tool name(s), in order*, ignoring params | **the gate** (`aggregate_reward_score`) |
  | `format_match` | right tool(s) **and** the exact harness array form | diagnostic |
- **Pick the gate to match what you're actually testing.** `toolset_match` is the gate because distillation's job is tool *selection* — so it's parsed **leniently** (format-agnostic) to avoid punishing free-form `q` wording or output-shape quirks. Formatting is tracked separately in `format_match` rather than conflated into the gate.
- **Separate dimensions you want to reason about.** Splitting selection from formatting is what makes the base-vs-student story legible: the base model usually knows the tools but emits a bare object inside `<think>` (high `toolset_match`, low `format_match`); the distilled student matches both.
- **Fail soft, never crash.** A scorer that raises on one odd sample can tank a whole job — parsing tolerates malformed/empty output and returns `0.0` instead of throwing.

### Build → test → register

Because the scorer is shipped to SageMaker (not run in this notebook), **test it locally first** — the next cell runs its unit tests so a scoring bug is caught in seconds rather than inside a managed job. Then register it as an Evaluator (a Hub Content ARN) and hand that ARN to `CustomScorerEvaluator`.

In [ ]:
# Validate the scorer locally before the next cell ships it to SageMaker, then show the source it validated.
import subprocess, sys
from pathlib import Path
from IPython.display import display, Markdown

REWARD_FN = "reward_functions/litmus_reward_function.py"
TEST_FN = "reward_functions/test_litmus_reward_function.py"

# Run the unit tests in a subprocess so a failure gives a clean verdict here (not a raw traceback).
res = subprocess.run([sys.executable, TEST_FN], capture_output=True, text=True)
print(res.stdout, end="")
if res.returncode == 0:
    print("\n" + "=" * 60 + "\n✅  UNIT TESTS PASSED — the scorer is safe to register.\n" + "=" * 60)
else:
    print(res.stderr[-1500:])
    print("\n" + "=" * 60 + "\n❌  UNIT TESTS FAILED — fix the reward function before registering.\n" + "=" * 60)

# Show the exact source that was validated (read from the file — no duplicated copy to drift).
display(Markdown(f"**Reward function** (`{REWARD_FN}`):\n\n```python\n{Path(REWARD_FN).read_text()}\n```"))

In [ ]:
# Register the (tested) reward function as an Evaluator version — a Hub Content ARN that CustomScorerEvaluator
# needs (not a raw Lambda ARN). This is a one-time AWS call; its SDK/registration logging is noisy, which is
# why it's its own cell.
from sagemaker.ai_registry.evaluator import Evaluator
from sagemaker.ai_registry.air_constants import REWARD_FUNCTION

evaluator = Evaluator.create(name="seqkd-litmus-toolcall", type=REWARD_FUNCTION, source=REWARD_FN)
EVALUATOR_ARN = evaluator.arn
print("Evaluator ARN:", EVALUATOR_ARN)

## 4. Run the managed evaluation jobs

One Custom Scorer job per task (product/shop/voucher) — there's no combined job; the aggregate is computed from the per-task scores later. With `EVALUATE_BASE=True`, each task launches **two** underlying serverless jobs (student + base comparison).

**Concurrency quota.** Those serverless jobs count against the account limit *"Maximum number of concurrent model evaluation serverless jobs per Region"* (quota `L-619D690E`); exceeding it fails a task with `SAGEMAKER_RESOURCE_LIMIT`. The cell reads that quota and schedules tasks to stay under it, falling back to one task at a time if the quota lookup fails. SDK log noise is suppressed only around the submit call and then restored (toggle with `QUIET_SDK_LOGS`).

In [ ]:
from litmus_utils import run_eval_jobs

# Quota-aware scheduler (see the section above). Returns the executions + each task's final status.
EXECUTIONS, results = run_eval_jobs(
    datasets=DATASETS, model_pkg_arn=MODEL_PKG_ARN, evaluator_arn=EVALUATOR_ARN,
    s3_output=S3_OUTPUT, region=REGION, mlflow_experiment=MLFLOW_EXPERIMENT_NAME,
    evaluate_base=EVALUATE_BASE)

## 5. Read the results and decide the gate

Pull the per-task metrics for student (and base) from each job's output and show two tables:

- **`toolset_match`** — did the model call the right tool(s), ignoring format? This is the **gate** metric.
- **`format_match`** — did it also emit the exact harness `[{...}]` form? Conformance only (the base model usually knows the tools but not the format, so its `toolset_match` ≫ `format_match`).

The **aggregate is a simple mean over tasks** (macro-average) — each intent counts equally so a small/hard one isn't overshadowed — matching the ASR notebook. **Gate:** if the student's aggregate `toolset_match` beats base, proceed to the full ASR eval; otherwise fix training first. Finally, a presigned link to the MLflow experiment for the run's lineage.

In [ ]:
from litmus_utils import read_task_metrics, print_results_tables, mlflow_experiment_link

per_task = read_task_metrics(datasets=DATASETS, s3_output=S3_OUTPUT, counts=COUNTS,
                             region=REGION, evaluate_base=EVALUATE_BASE)

# toolset_match = did it call the right tool(s), format-agnostic (the GATE); format_match = exact harness form.
agg = print_results_tables(per_task, EVAL_TASKS, metrics=[
    ("toolset_match", "toolset_match — tool-selection accuracy (format-agnostic)  [GATE]"),
    ("format_match",  "format_match — exact `<tool_call>[{...}]</tool_call>` conformance")])

# ---- The gate: the student must beat base on aggregate toolset_match to justify the expensive ASR eval ----
as_, ab_ = agg["toolset_match"]["student"], agg["toolset_match"]["base"]
fs_, fb_ = agg["format_match"]["student"], agg["format_match"]["base"]
print("\n--- Gate decision (mean-over-tasks aggregate toolset_match) ---")
if as_ is None:
    print("Could not read per-task 'toolset_match'; inspect the tables above.")
elif ab_ is None:
    print(f"student {as_:.1%} (no base to compare; set EVALUATE_BASE=True).")
elif as_ > ab_:
    print(f"✅ PASS  student {as_:.1%} > base {ab_:.1%} — proceed to the full ASR eval.")
    if fb_ is not None and fs_ is not None:
        print(f"   note: tool selection gap is +{100*(as_-ab_):.1f} pts; exact-format gap is "
              f"+{100*(fs_-fb_):.1f} pts (base knows the tools but not the harness format).")
else:
    print(f"❌ FAIL  student {as_:.1%} <= base {ab_:.1%} — fix training before the heavy eval.")

mlflow_experiment_link(MLFLOW_EXPERIMENT_NAME, sagemaker_session)